# PRL 123, 116104 (2019): Fig. 2(a–c) TE-like bands with legume

First-pass reproduction of the TE-like photonic bands only. The paper used a finite-element method; the `legume` GME cutoff and basis below are our own numerical choices. This notebook does not compute polarization ellipses, Stokes parameters, Q factors, RCWA spectra, or C points.

## 1. Imports and local environment

In [1]:
from pathlib import Path
import csv
import importlib.metadata as metadata
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import legume

OUT = Path.cwd()
FIG_DIR = OUT / 'figures'
DATA_DIR = OUT / 'data'
FIG_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

LEGUME_VERSION = metadata.version('legume-gme')
print('Python:', platform.python_version())
print('legume-gme:', LEGUME_VERSION)
print('legume module:', legume.__file__)
print('Actual polygon API:', legume.Poly)
print('Actual square API:', legume.Square)
print('Actual GME API:', legume.GuidedModeExp)

Python: 3.12.14
legume-gme: 1.0.3
legume module: /workspace/legume/legume/__init__.py
Actual polygon API: <class 'legume.phc.shapes.Poly'>
Actual square API: <class 'legume.phc.shapes.Square'>
Actual GME API: <class 'legume.gme.gme.GuidedModeExp'>


## 2. Paper parameters in normalized units ($a=1$)

In [2]:
a_nm = 450.0
t_nm = 100.0
n_slab = 2.02
eps_slab = n_slab**2
eps_air = 1.0
side0_nm = 247.0

t = t_nm / a_nm
side0 = side0_nm / a_nm
S0 = side0**2
alphas = (0.0, 0.1, 1.0)

print(f't/a = {t:.9f}')
print(f'n = {n_slab:.3f}, epsilon = {eps_slab:.6f}')
print(f'S0/a^2 = {S0:.9f}')

t/a = 0.222222222
n = 2.020, epsilon = 4.080400
S0/a^2 = 0.301279012


## 3. Square lattice

In [3]:
lattice = legume.Lattice('square')
print(lattice)
print('a1 =', lattice.a1, 'a2 =', lattice.a2)

Lattice(a1 = [1.0000, 0.0000], a2 = [0.0000, 1.0000])
a1 = [1 0] a2 = [0 1]


## 4–5. Freestanding slab and area-preserving hole geometry

Vertices passed to `legume.Poly` are counter-clockwise, as required by legume 1.0.3. The trapezoid is centered in the cell, has its bottom edge at $y=-h/2$, and keeps the $x\to -x$ mirror.

In [4]:
def hole_dimensions(alpha):
    if not 0.0 <= alpha <= 1.0:
        raise ValueError('alpha must lie in [0, 1]')
    L = np.sqrt(2.0 * S0 / (2.0 - alpha))
    h = L
    L_top = L * (1.0 - alpha)
    return L, h, L_top

def make_hole(alpha):
    L, h, L_top = hole_dimensions(alpha)
    if np.isclose(alpha, 0.0):
        return legume.Square(eps=eps_air, x_cent=0.0, y_cent=0.0, a=L)
    if np.isclose(alpha, 1.0):
        return legume.Poly(
            eps=eps_air,
            x_edges=[-L/2, L/2, 0.0],
            y_edges=[-h/2, -h/2, h/2],
        )
    return legume.Poly(
        eps=eps_air,
        x_edges=[-L/2, L/2, L_top/2, -L_top/2],
        y_edges=[-h/2, -h/2, h/2, h/2],
    )

def make_phc(alpha):
    phc = legume.PhotCryst(lattice, eps_l=eps_air, eps_u=eps_air)
    phc.add_layer(d=t, eps_b=eps_slab)
    phc.add_shape(make_hole(alpha))
    return phc

geometry_rows = []
for alpha in alphas:
    L, h, L_top = hole_dimensions(alpha)
    geometry_rows.append({
        'alpha': alpha, 'L/a': L, 'h/a': h, 'L_top/a': L_top,
        'L (nm)': L*a_nm, 'h (nm)': h*a_nm, 'L_top (nm)': L_top*a_nm,
        'area/a^2': L*(L + L_top)/2,
    })
geometry_table = pd.DataFrame(geometry_rows)
geometry_table

,alpha,L/a,h/a,L_top/a,L (nm),h (nm),L_top (nm),area/a^2
0,0.0,0.548889,0.548889,0.548889,247.000000,247.000000,247.000000,0.301279
1,0.1,0.563148,0.563148,0.506833,253.416653,253.416653,228.074988,0.301279
2,1.0,0.776246,0.776246,0.000000,349.310750,349.310750,0.000000,0.301279


## 6. Unit-cell geometry check

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.2), constrained_layout=True)
for ax, alpha in zip(axes, alphas):
    phc = make_phc(alpha)
    legume.viz.eps_xy(phc, z=t/2, Nx=301, Ny=301, ax=ax, cmap='Blues', cbar=False)
    ax.set_aspect('equal')
    ax.set_xlabel('$x/a$')
    ax.set_ylabel('$y/a$')
    ax.set_title(rf'$\alpha={alpha:g}$')
    single = plt.figure(figsize=(3.4, 3.2))
    single_ax = single.add_subplot(111)
    legume.viz.eps_xy(phc, z=t/2, Nx=301, Ny=301, ax=single_ax, cmap='Blues', cbar=False)
    single_ax.set_aspect('equal')
    single_ax.set_xlabel('$x/a$'); single_ax.set_ylabel('$y/a$')
    single_ax.set_title(rf'$\alpha={alpha:g}$')
    tag = str(alpha).replace('.', 'p')
    single.savefig(FIG_DIR / f'unit_cell_alpha_{tag}.png', dpi=180, bbox_inches='tight')
    plt.close(single)
fig.savefig(FIG_DIR / 'unit_cells_alpha_0_0p1_1.png', dpi=180, bbox_inches='tight')
plt.show()

## 7–8. $X'\rightarrow\Gamma\rightarrow X$ path

The signed plotting coordinate is $ka/(2\pi)$. The left segment follows $+k_y$ toward $\Gamma$; the right segment follows $+k_x$ away from $\Gamma$. There are 101 total points.

In [6]:
k_reduced = np.linspace(-0.1, 0.1, 101)
kpoints = np.zeros((2, k_reduced.size))
left = k_reduced < 0
right = k_reduced > 0
kpoints[1, left] = -2*np.pi*k_reduced[left]  # X' (+ky) -> Gamma
kpoints[0, right] = 2*np.pi*k_reduced[right] # Gamma -> X (+kx)
gamma_index = int(np.argmin(np.abs(k_reduced)))
print('kpoints shape:', kpoints.shape, 'Gamma index:', gamma_index)
print('endpoints (ka/2pi):', kpoints[:, 0]/(2*np.pi), kpoints[:, -1]/(2*np.pi))

kpoints shape: (2, 101) Gamma index: 50
endpoints (ka/2pi): [0.  0.1] [0.1 0. ]


## 9–11. TE-like GuidedModeExp calculation

For an up-down symmetric slab, legume documents `[0, 3]` as an xy-even (TE-like) guided-mode basis. All eight computed eigenvalues are retained. `compute_im=False` keeps this stage strictly to real band frequencies.

In [7]:
GME_PARAMS = dict(
    gmax=4.5,
    truncate_g='abs',
    gmode_inds=[0, 3],
    numeig=8,
    compute_im=False,
    verbose=False,
)
GME_PARAMS

{'gmax': 4.5,
 'truncate_g': 'abs',
 'gmode_inds': [0, 3],
 'numeig': 8,
 'compute_im': False,
 'verbose': False}

In [8]:
def run_bands(alpha, kpts, gmax=GME_PARAMS['gmax']):
    gme = legume.GuidedModeExp(make_phc(alpha), gmax=gmax, truncate_g=GME_PARAMS['truncate_g'])
    gme.run(
        kpoints=kpts,
        gmode_inds=GME_PARAMS['gmode_inds'],
        numeig=GME_PARAMS['numeig'],
        compute_im=GME_PARAMS['compute_im'],
        verbose=GME_PARAMS['verbose'],
    )
    return gme

results = {}
for alpha in alphas:
    print(f'Running alpha={alpha:g} ...')
    results[alpha] = run_bands(alpha, kpoints)
    print('  done:', results[alpha].freqs.shape)

Running alpha=0 ...


  done: (101, 8)
Running alpha=0.1 ...


  done: (101, 8)
Running alpha=1 ...


  done: (101, 8)


## 12–14. Side-by-side band structures

Band identity is assessed after inspecting the $\Gamma$ table. The candidate corresponding to the paper's TE2 is the lowest nonzero branch in the plotted frequency window: it lies near 0.75 at $\Gamma$, is isolated from the next branch near 0.78, and follows the lowest bold branch in Fig. 2(a–c). This happens to be numerical column 1 here, but the assignment is based on frequency ordering and dispersion, not the column number. All bands remain visible.

In [9]:
gamma_tables = {}
for alpha in alphas:
    vals = np.asarray(results[alpha].freqs[gamma_index])
    gamma_tables[alpha] = pd.DataFrame({
        'numerical_column': np.arange(vals.size),
        'a/lambda at Gamma': vals,
        'in_initial_plot_window': (vals >= 0.68) & (vals <= 0.86),
    })
    print(f'alpha={alpha:g}')
    display(gamma_tables[alpha])

TARGET_COLUMN = 1
target_gamma = {alpha: float(results[alpha].freqs[gamma_index, TARGET_COLUMN]) for alpha in alphas}
print('Identified TE2 candidate at Gamma:', target_gamma)

alpha=0


,numerical_column,a/lambda at Gamma,in_initial_plot_window
0,0,4.499782e-08,False
1,1,7.490528e-01,True
2,2,7.787555e-01,True
3,3,7.935810e-01,True
4,4,7.935810e-01,True
5,5,9.992048e-01,False
6,6,1.062070e+00,False
7,7,1.068438e+00,False


alpha=0.1


,numerical_column,a/lambda at Gamma,in_initial_plot_window
0,0,2.990466e-08,False
1,1,7.488224e-01,True
2,2,7.787683e-01,True
3,3,7.909108e-01,True
4,4,7.965800e-01,True
5,5,1.000018e+00,False
6,6,1.061439e+00,False
7,7,1.067820e+00,False


alpha=1


,numerical_column,a/lambda at Gamma,in_initial_plot_window
0,0,9.921087e-09,False
1,1,7.364318e-01,True
2,2,7.828280e-01,True
3,3,7.964541e-01,True
4,4,7.976814e-01,True
5,5,1.032523e+00,False
6,6,1.032714e+00,False
7,7,1.050128e+00,False


Identified TE2 candidate at Gamma: {0.0: 0.7490528285225992, 0.1: 0.7488224018456732, 1.0: 0.7364318087146989}


In [10]:
fig, axes = plt.subplots(1, 3, figsize=(10.8, 4.0), sharey=True, constrained_layout=True)
for ax, alpha in zip(axes, alphas):
    freqs = np.asarray(results[alpha].freqs)
    for band in range(freqs.shape[1]):
        if band == TARGET_COLUMN:
            ax.plot(k_reduced, freqs[:, band], color='0.08', lw=2.4, label='TE2 candidate')
        else:
            ax.plot(k_reduced, freqs[:, band], color='0.55', lw=1.0)
    ax.axvline(0, color='0.82', lw=0.8)
    ax.set_xlim(-0.1, 0.1); ax.set_ylim(0.68, 0.86)
    ax.set_xticks([-0.1, 0.0, 0.1], ["$X'$", '$\Gamma$', '$X$'])
    ax.set_xlabel(r'$ka/(2\pi)$')
    ax.set_title(rf'$\alpha={alpha:g}$')
axes[0].set_ylabel(r'$a/\lambda$')
axes[0].legend(loc='upper left', frameon=False, fontsize=8)
fig.savefig(FIG_DIR / 'Fig2_abc_TE_like_bands.png', dpi=220, bbox_inches='tight')
plt.show()

<>:11: SyntaxWarning: invalid escape sequence '\G'
<>:11: SyntaxWarning: invalid escape sequence '\G'
/tmp/ipykernel_1382/1405842707.py:11: SyntaxWarning: invalid escape sequence '\G'
  ax.set_xticks([-0.1, 0.0, 0.1], ["$X'$", '$\Gamma$', '$X$'])


## 15–17. Save raw arrays and tabular data

In [11]:
npz_payload = {'kpoints': kpoints, 'k_reduced': k_reduced, 'gamma_index': gamma_index}
for alpha in alphas:
    tag = str(alpha).replace('.', 'p')
    freqs = np.asarray(results[alpha].freqs)
    npz_payload[f'freqs_alpha_{tag}'] = freqs
    frame = pd.DataFrame({'ka_over_2pi': k_reduced, 'kx': kpoints[0], 'ky': kpoints[1]})
    for band in range(freqs.shape[1]):
        frame[f'band_{band}_a_over_lambda'] = freqs[:, band]
    frame.to_csv(DATA_DIR / f'bands_alpha_{tag}.csv', index=False)
np.savez_compressed(DATA_DIR / 'Fig2_abc_TE_like_raw.npz', **npz_payload)
geometry_table.to_csv(DATA_DIR / 'hole_dimensions.csv', index=False)
pd.concat(gamma_tables, names=['alpha']).to_csv(DATA_DIR / 'gamma_eigenfrequencies.csv')
print('Saved data files:')
for path in sorted(DATA_DIR.iterdir()): print(' ', path.name)

Saved data files:
  Fig2_abc_TE_like_raw.npz
  bands_alpha_0p0.csv
  bands_alpha_0p1.csv
  bands_alpha_1p0.csv
  gamma_eigenfrequencies.csv
  hole_dimensions.csv


## C. First convergence check at $\Gamma$

This inexpensive check uses $g_{max}=3.5,4.0,4.5,5.0$ for all three geometries. At each cutoff, the target is matched to the closest $\Gamma$ frequency from the baseline run; this avoids assuming that a fixed raw column is universally meaningful. A full overlap-based band tracker can be added in a later precision study.

In [12]:
gmax_values = (3.5, 4.0, 4.5, 5.0)
conv_rows = []
gamma_kpoint = np.zeros((2, 1))
for alpha in alphas:
    reference = target_gamma[alpha]
    for gmax in gmax_values:
        gme = run_bands(alpha, gamma_kpoint, gmax=gmax)
        vals = np.asarray(gme.freqs[0])
        matched_column = int(np.argmin(np.abs(vals - reference)))
        conv_rows.append({
            'alpha': alpha, 'gmax': gmax, 'matched_column': matched_column,
            'TE2_candidate_a/lambda': vals[matched_column],
            'delta_from_gmax_4.5': vals[matched_column] - reference,
        })
convergence_table = pd.DataFrame(conv_rows)
convergence_table.to_csv(DATA_DIR / 'gmax_convergence_gamma_TE2_candidate.csv', index=False)
display(convergence_table)

Plane waves used in the expansion = 49.
Plane waves used in the expansion = 81.
Plane waves used in the expansion = 49.
Plane waves used in the expansion = 81.
Plane waves used in the expansion = 49.
Plane waves used in the expansion = 81.


,alpha,gmax,matched_column,TE2_candidate_a/lambda,delta_from_gmax_4.5
0,0.0,3.5,1,0.747217,-0.001836
1,0.0,4.0,1,0.748787,-0.000266
2,0.0,4.5,1,0.749053,0.000000
3,0.0,5.0,1,0.749336,0.000283
4,0.1,3.5,1,0.746970,-0.001853
5,0.1,4.0,1,0.748469,-0.000354
6,0.1,4.5,1,0.748822,0.000000
7,0.1,5.0,1,0.749088,0.000266
8,1.0,3.5,1,0.735697,-0.000735
9,1.0,4.0,1,0.735943,-0.000489


In [13]:
fig, ax = plt.subplots(figsize=(6.0, 3.8))
for alpha in alphas:
    sub = convergence_table[convergence_table['alpha'] == alpha]
    ax.plot(sub['gmax'], sub['TE2_candidate_a/lambda'], 'o-', label=rf'$\alpha={alpha:g}$')
ax.set_xlabel('$g_{max}$'); ax.set_ylabel(r'TE2 candidate $a/\lambda$ at $\Gamma$')
ax.grid(alpha=0.25); ax.legend(frameon=False)
fig.savefig(FIG_DIR / 'gmax_convergence_gamma_TE2_candidate.png', dpi=200, bbox_inches='tight')
plt.show()

## Short numerical summary

In [14]:
summary = pd.DataFrame({
    'alpha': alphas,
    'L/a': [hole_dimensions(a)[0] for a in alphas],
    'L_top/a': [hole_dimensions(a)[2] for a in alphas],
    'TE2 candidate a/lambda at Gamma': [target_gamma[a] for a in alphas],
})
display(summary)
print(f'legume-gme {LEGUME_VERSION}; GME settings: {GME_PARAMS}')
print('Trend check: the candidate branch remains the lowest band in the paper window and shifts downward as alpha grows,')
print('consistent with Fig. 2(a-c) at first-pass qualitative level. Remaining absolute differences should first be')
print('tested against GME cutoff/basis convergence, then refractive index dispersion and exact band identification/geometry convention.')

,alpha,L/a,L_top/a,TE2 candidate a/lambda at Gamma
0,0.0,0.548889,0.548889,0.749053
1,0.1,0.563148,0.506833,0.748822
2,1.0,0.776246,0.000000,0.736432


legume-gme 1.0.3; GME settings: {'gmax': 4.5, 'truncate_g': 'abs', 'gmode_inds': [0, 3], 'numeig': 8, 'compute_im': False, 'verbose': False}
Trend check: the candidate branch remains the lowest band in the paper window and shifts downward as alpha grows,
consistent with Fig. 2(a-c) at first-pass qualitative level. Remaining absolute differences should first be
tested against GME cutoff/basis convergence, then refractive index dispersion and exact band identification/geometry convention.
